# SECCIÓN 3. PREPARACIÓN DE DATOS

Este cuaderno desarrolla los puntos 3 y 4 del trabajo mediante la metodología CRISP-DM. El objetivo es predecir `trip_duration` (segundos) con información disponible al inicio del viaje, evitando fuga de información.


## 3.1 Selección de variables desde el conocimiento del negocio

Se excluyen identificadores, datos personales y variables sin relación operativa (`RecordID`, `FullName`, `Phone`, `ZodiacSign`, `FavoriteColor`, `Hobby`). También se elimina `total_fare`, porque se conoce durante o después del viaje y produciría fuga de información. `VendorID` se descarta por ser casi constante. Se conserva `trip_distance_miles` y se elimina `trip_distance`: ambas representan la misma magnitud, pero ocho valores erróneos de la segunda columna (−5, 500 y 800 km) destruyen su correlación, mientras que el 99,5 % de las filas comparables conserva correctamente la conversión.


In [1]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Funciona directamente en Colab y también junto al notebook descargado.
file_path = '/content/Taxi_Trip_Duration.csv'
if not os.path.exists(file_path):
    file_path = 'Taxi_Trip_Duration.csv'

df = pd.read_csv(file_path)
print(f'Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas')
display(df.head())


Dimensiones iniciales: 1800 filas y 21 columnas


,RecordID,FullName,Phone,ZodiacSign,FavoriteColor,Hobby,pickup_datetime,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,trip_distance,trip_distance_miles,hour,day_of_week,is_weekend,weather,VendorID,trip_duration,total_fare
0,REC00001,Sofia Miller,3.449545e+09,Cancer,Yellow,Cooking,01/06/2023 00:00:00,40.660618,-73.979191,40.669406,-74.003883,2.0,NaN,9.091601,21,4,0,Fog,A,2296.617526,40.229624
1,REC00002,John Gonzalez,3.648211e+09,Capricorn,White,Gaming,01/06/2023 00:17:00,40.717049,-73.911793,40.724527,-73.918776,2.0,11.823875,7.347013,22,4,0,Rain,A,1861.815211,32.842164
2,REC00003,Thomas Perez,3.374371e+09,Aries,Green,Reading,01/06/2023 00:34:00,NaN,-74.006844,40.716247,-74.008732,1.0,5.286519,3.284890,9,1,0,Clear,A,838.891983,15.233691
3,REC00004,Joseph Thomas,3.697255e+09,Libra,Yellow,Hiking,01/06/2023 00:51:00,NaN,-73.971136,40.751094,-73.951873,1.0,5.120515,3.181740,18,3,0,Snow,A,1223.455139,19.487708
4,REC00005,Charles Lopez,3.432576e+09,Sagittarius,Red,Reading,01/06/2023 01:08:00,40.817390,-74.055283,40.835754,-73.997030,NaN,2.633424,1.636333,18,1,0,Clear,A,646.133860,11.904101


In [2]:
# Auditoría de redundancia y calidad de la distancia
valid_dist = df['trip_distance'].notna() & df['trip_distance_miles'].notna()
expected_km = df['trip_distance_miles'] * 1.609344
bad_dist = valid_dist & (df['trip_distance'].sub(expected_km).abs() > 0.02)
conversion_ok = 100 * (valid_dist.sum() - bad_dist.sum()) / valid_dist.sum()

print(f'Correlación original km-millas: {df.trip_distance.corr(df.trip_distance_miles):.4f}')
print(f'Registros inconsistentes: {bad_dist.sum()}')
print(f'Conversión correcta entre filas comparables: {conversion_ok:.2f}%')
display(df.loc[bad_dist, ['trip_distance', 'trip_distance_miles', 'trip_duration']])

exclude = ['RecordID','FullName','Phone','ZodiacSign','FavoriteColor','Hobby',
           'total_fare','VendorID','trip_distance']
df_clean = df.drop(columns=exclude, errors='ignore').copy()
print('Variables inicialmente conservadas:', df_clean.columns.tolist())


Correlación original km-millas: 0.1102
Registros inconsistentes: 8
Conversión correcta entre filas comparables: 99.51%


,trip_distance,trip_distance_miles,trip_duration
739,500.0,1.683960,754.632193
948,500.0,5.091247,1540.159587
963,800.0,5.352428,1204.926759
1024,-5.0,1.908915,522.371788
1189,-5.0,2.225154,487.526546
1203,800.0,2.238030,664.217448
1650,500.0,3.027419,728.600851
1660,-5.0,3.318510,1141.237628


Variables inicialmente conservadas: ['pickup_datetime', 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'trip_distance_miles', 'hour', 'day_of_week', 'is_weekend', 'weather', 'trip_duration']


## 3.2 Descripción estadística

Se revisan tipos, medidas descriptivas, valores faltantes, duplicados y distribución de la variable objetivo. Esto permite distinguir errores de captura de observaciones extremas plausibles.


In [3]:
print('Tipos de datos:')
display(df.dtypes.to_frame('tipo'))
print('Descripción numérica:')
display(df.describe().T)
print('Descripción categórica:')
display(df.describe(include=['object']).T)

missing = pd.DataFrame({'nulos': df.isna().sum(), 'porcentaje': df.isna().mean()*100})
display(missing[missing.nulos > 0].sort_values('porcentaje', ascending=False))
print('Filas duplicadas:', df.duplicated().sum())
print(f'Asimetría inicial de trip_duration: {df.trip_duration.skew():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(df.trip_duration, bins=50); ax[0].set_title('trip_duration original'); ax[0].set_xlabel('segundos')
ax[1].hist(np.log1p(df.trip_duration), bins=50, color='orange'); ax[1].set_title('log(1 + trip_duration)')
plt.tight_layout(); plt.show()


Tipos de datos:


,tipo
RecordID,object
FullName,object
Phone,float64
ZodiacSign,object
FavoriteColor,object
Hobby,object
pickup_datetime,object
pickup_latitude,float64
pickup_longitude,float64
dropoff_latitude,float64


Descripción numérica:


,count,mean,std,min,25%,50%,75%,max
Phone,1800.0,3.502309e+09,2.896287e+08,3.000030e+09,3.257613e+09,3.497885e+09,3.747475e+09,3.999212e+09
pickup_latitude,1620.0,4.074959e+01,5.007573e-02,4.055915e+01,4.071596e+01,4.074955e+01,4.078272e+01,4.093801e+01
pickup_longitude,1800.0,-7.397858e+01,4.986388e-02,-7.414992e+01,-7.401224e+01,-7.397975e+01,-7.394396e+01,-7.383761e+01
dropoff_latitude,1800.0,4.074947e+01,5.918821e-02,4.052061e+01,4.071040e+01,4.074950e+01,4.078771e+01,4.095076e+01
dropoff_longitude,1800.0,-7.397925e+01,5.941080e-02,-7.417157e+01,-7.402002e+01,-7.397981e+01,-7.393925e+01,-7.378579e+01
passenger_count,1628.0,2.030713e+00,1.609832e+00,0.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,2.000000e+01
trip_distance,1628.0,6.944888e+00,3.518067e+01,-5.000000e+00,2.471161e+00,4.269459e+00,6.982435e+00,8.000000e+02
trip_distance_miles,1800.0,3.144034e+00,2.219837e+00,2.633679e-02,1.503807e+00,2.631996e+00,4.321675e+00,1.483912e+01
hour,1800.0,1.155222e+01,6.931456e+00,0.000000e+00,6.000000e+00,1.200000e+01,1.800000e+01,2.300000e+01
day_of_week,1800.0,3.022778e+00,2.006945e+00,0.000000e+00,1.000000e+00,3.000000e+00,5.000000e+00,6.000000e+00


Descripción categórica:


,count,unique,top,freq
RecordID,1800,1800,REC00001,1
FullName,1800,393,Robert Garcia,11
ZodiacSign,1800,12,Taurus,179
FavoriteColor,1800,10,Gray,197
Hobby,1800,10,Gardening,194
pickup_datetime,1800,1800,01/06/2023 00:00:00,1
weather,1620,4,Clear,955
VendorID,1800,3,A,1749


,nulos,porcentaje
pickup_latitude,180,10.000000
weather,180,10.000000
passenger_count,172,9.555556
trip_distance,172,9.555556


Filas duplicadas: 0
Asimetría inicial de trip_duration: 20.7623


## 3.3 Limpieza de atípicos

Se aplican reglas de negocio transparentes: duración entre 60 segundos y 4 horas, distancia positiva y no mayor de 100 millas, y entre 1 y 6 pasajeros enteros. Los nulos de pasajeros se conservan temporalmente para imputarlos dentro del pipeline. Esta decisión elimina errores evidentes sin aplicar recortes estadísticos arbitrarios.


In [4]:
before = len(df_clean)
rule_duration = df_clean.trip_duration.between(60, 14400)
rule_distance = df_clean.trip_distance_miles.gt(0) & df_clean.trip_distance_miles.le(100)
rule_passenger = (df_clean.passenger_count.isna() |
                  (df_clean.passenger_count.between(1,6) & df_clean.passenger_count.mod(1).eq(0)))

audit = pd.DataFrame({
    'regla': ['Duración fuera de 60–14.400 s', 'Distancia fuera de 0–100 millas', 'Pasajeros fuera de 1–6 enteros'],
    'registros_detectados': [(~rule_duration).sum(), (~rule_distance).sum(), (~rule_passenger).sum()]
})
display(audit)
df_clean = df_clean[rule_duration & rule_distance & rule_passenger].copy()
print(f'Antes: {before}; después: {len(df_clean)}; eliminados: {before-len(df_clean)} ({100*(before-len(df_clean))/before:.2f}%)')


,regla,registros_detectados
0,Duración fuera de 60–14.400 s,8
1,Distancia fuera de 0–100 millas,0
2,Pasajeros fuera de 1–6 enteros,8


Antes: 1800; después: 1784; eliminados: 16 (0.89%)


## 3.4 Limpieza de nulos

La partición 70/30 se realiza antes de aprender cualquier imputación. En cada pliegue de validación cruzada, las variables numéricas se imputan con la mediana y `weather` con la categoría `Desconocido`. Así, los estadísticos del conjunto de prueba nunca participan en el entrenamiento.

## 3.5 Creación de nuevas variables y One-Hot Encoding

`pickup_datetime` se convierte desde el formato día/mes/año. A partir de ella se crean `pickup_hour`, `pickup_day_of_week` y `pickup_is_weekend`. Se sustituyen las columnas temporales originales porque no coinciden con la fecha (la hora solo coincide en cerca del 4,6 % de los registros). No se crea el mes porque todos los registros pertenecen a junio y sería una variable constante. `weather` se transforma mediante One-Hot Encoding dentro del pipeline.


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

dt = pd.to_datetime(df_clean['pickup_datetime'], dayfirst=True, errors='coerce')
print('Fechas inválidas:', dt.isna().sum())
print(f'Coincidencia de hour original con fecha: {(df_clean.hour == dt.dt.hour).mean()*100:.2f}%')
print(f'Coincidencia de day_of_week original con fecha: {(df_clean.day_of_week == dt.dt.dayofweek).mean()*100:.2f}%')

df_clean['pickup_hour'] = dt.dt.hour
df_clean['pickup_day_of_week'] = dt.dt.dayofweek
df_clean['pickup_is_weekend'] = dt.dt.dayofweek.isin([5,6]).astype(int)
df_clean = df_clean.drop(columns=['pickup_datetime','hour','day_of_week','is_weekend'])

X = df_clean.drop(columns='trip_duration')
y = df_clean['trip_duration']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=42)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Desconocido')),
                     ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

print('Entrenamiento:', X_train.shape, 'Prueba:', X_test.shape)
print('Numéricas:', num_cols); print('Categóricas:', cat_cols)


Fechas inválidas: 0
Coincidencia de hour original con fecha: 4.60%
Coincidencia de day_of_week original con fecha: 14.41%
Entrenamiento: (1248, 10) Prueba: (536, 10)
Numéricas: ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'trip_distance_miles', 'pickup_hour', 'pickup_day_of_week', 'pickup_is_weekend']
Categóricas: ['weather']


## 3.6 Análisis de relaciones

El análisis lineal utiliza Pearson. Para detectar asociaciones monótonas se utiliza Spearman y para relaciones no lineales se calcula información mutua sobre los datos de entrenamiento ya imputados y codificados. `trip_distance_miles` es la señal predictiva principal; las demás variables aportan patrones temporales, espaciales y climáticos complementarios.


In [6]:
from sklearn.feature_selection import mutual_info_regression

explore = X_train.select_dtypes(include=np.number).copy()
explore['trip_duration'] = y_train
pearson = explore.corr(method='pearson').trip_duration.drop('trip_duration').sort_values(key=abs, ascending=False)
spearman = explore.corr(method='spearman').trip_duration.drop('trip_duration').sort_values(key=abs, ascending=False)
display(pd.DataFrame({'Pearson': pearson, 'Spearman': spearman}))

X_mi = preprocessor.fit_transform(X_train)
if hasattr(X_mi, 'toarray'): X_mi = X_mi.toarray()
feature_names = preprocessor.get_feature_names_out()
mi = pd.Series(mutual_info_regression(X_mi, y_train, random_state=42), index=feature_names).sort_values(ascending=False)
print('Información mutua (relaciones no lineales):')
display(mi.head(12).to_frame('información_mutua'))

plt.figure(figsize=(7,4)); plt.scatter(X_train.trip_distance_miles, y_train, alpha=.35)
plt.xlabel('Distancia (millas)'); plt.ylabel('Duración (s)'); plt.title('Relación distancia-duración'); plt.show()


,Pearson,Spearman
dropoff_latitude,0.007962,-0.013471
dropoff_longitude,0.009093,-0.014495
passenger_count,0.001809,-0.015445
pickup_day_of_week,-0.026685,-0.041484
pickup_hour,0.015554,0.012960
pickup_is_weekend,-0.033626,-0.042715
pickup_latitude,0.003404,-0.005771
pickup_longitude,0.018440,-0.004924
trip_distance_miles,0.891182,0.852557


Información mutua (relaciones no lineales):


,información_mutua
num__trip_distance_miles,0.751193
num__dropoff_latitude,0.034680
cat__weather_Clear,0.031046
cat__weather_Snow,0.030323
num__pickup_is_weekend,0.019119
cat__weather_Fog,0.016269
cat__weather_Rain,0.004737
num__pickup_latitude,0.003928
num__passenger_count,0.001197
num__pickup_longitude,0.000000


## 3.7 Reducción de dimensiones

PCA se evalúa únicamente sobre las variables numéricas de entrenamiento escaladas. Se reporta cuántos componentes hacen falta para explicar al menos el 90 % de la varianza. No se incorpora al modelo final porque el número de predictores es pequeño, no existe multicolinealidad severa después de eliminar la distancia redundante y conservar las variables facilita la interpretación.


In [7]:
from sklearn.decomposition import PCA

X_num = num_pipe.fit_transform(X_train[num_cols])
pca = PCA().fit(X_num)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n90 = np.argmax(cumvar >= .90) + 1
print(f'Primer componente: {pca.explained_variance_ratio_[0]*100:.2f}%')
print(f'Componentes para explicar al menos 90%: {n90} de {len(num_cols)}')
plt.plot(range(1,len(cumvar)+1), cumvar, marker='o'); plt.axhline(.90,color='red',ls='--')
plt.xlabel('Componentes'); plt.ylabel('Varianza acumulada'); plt.title('Evaluación de PCA'); plt.show()


Primer componente: 21.48%
Componentes para explicar al menos 90%: 6 de 9


## 3.8 Balanceo

No aplica. SMOTE, sobremuestreo y submuestreo están diseñados para clases categóricas; `trip_duration` es una variable continua y el ejercicio corresponde a regresión.

# SECCIÓN 4. MODELAMIENTO Y EVALUACIÓN

## 4.1 Configuración y selección de métodos de Machine Learning

### 4.1.1 Cinco modelos clásicos

Regresión lineal (línea base interpretable), KNN (vecindades), árbol de decisión (reglas no lineales), SVR (margen y kernel) y MLP (red neuronal).

### 4.1.2 Tres ensambles

VotingRegressor (votación/promedio), BaggingRegressor (bagging) e HistGradientBoostingRegressor (boosting). Todos se integran con el mismo pipeline de preprocesamiento.


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import VotingRegressor, BaggingRegressor, HistGradientBoostingRegressor

classic_models = {
 'Regresión lineal': LinearRegression(),
 'KNN': KNeighborsRegressor(),
 'Árbol de decisión': DecisionTreeRegressor(random_state=42),
 'SVR': SVR(),
 'MLP': MLPRegressor(random_state=42, max_iter=1000, early_stopping=True)
}
ensemble_models = {
 'Voting': VotingRegressor([('lr',LinearRegression()),
                            ('tree',DecisionTreeRegressor(max_depth=5,random_state=42)),
                            ('hgb',HistGradientBoostingRegressor(random_state=42))]),
 'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), random_state=42),
 'Boosting': HistGradientBoostingRegressor(random_state=42)
}
models = {**classic_models, **ensemble_models}
display(pd.DataFrame({'grupo': ['Clásico']*5+['Ensamble']*3,
                      'modelo': list(models.keys())}))


,grupo,modelo
0,Clásico,Regresión lineal
1,Clásico,KNN
2,Clásico,Árbol de decisión
3,Clásico,SVR
4,Clásico,MLP
5,Ensamble,Voting
6,Ensamble,Bagging
7,Ensamble,Boosting


## 4.2 Ajuste de hiperparámetros con el 70 % de los datos haciendo cross validation

### 4.2.1 Justificación de la métrica

MAE es la métrica principal porque expresa el error directamente en segundos y es menos sensible a errores grandes aislados que RMSE. También se reportan RMSE, que penaliza fuertemente errores grandes, y R², que muestra la proporción de variabilidad explicada. MAPE no se usa porque las duraciones pequeñas pueden inflar artificialmente el porcentaje.

### 4.2.2 Ajuste de modelos clásicos

Se utiliza GridSearchCV con cinco pliegues únicamente sobre el 70 % de entrenamiento. Se ajustan KNN, árbol, SVR y MLP; la regresión lineal sirve como línea base sin hiperparámetros relevantes.

### 4.2.3 Ajuste de dos modelos de ensamble

Se ajustan Bagging y Boosting, tal como pide el enunciado. Voting se evalúa con una combinación previamente definida para representar el ensamble de votación.


In [9]:
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

param_grids = {
 'KNN': {'model__n_neighbors':[3,5,7,10,15], 'model__weights':['uniform','distance'], 'model__p':[1,2]},
 'Árbol de decisión': {'model__max_depth':[5,10,15,None], 'model__min_samples_split':[2,5,10], 'model__min_samples_leaf':[1,2,4]},
 'SVR': {'model__C':[1,10,50,100], 'model__epsilon':[.05,.1,.2], 'model__kernel':['linear','rbf']},
 'MLP': {'model__hidden_layer_sizes':[(50,),(100,),(50,50)], 'model__alpha':[.0001,.001], 'model__learning_rate_init':[.001,.01]},
 'Bagging': {'model__n_estimators':[20,50,100], 'model__max_samples':[.5,.75,1.0], 'model__max_features':[.7,1.0]},
 'Boosting': {'model__learning_rate':[.03,.05,.1], 'model__max_iter':[100,200], 'model__max_leaf_nodes':[15,31], 'model__l2_regularization':[0,.1]}
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
best_models, cv_rows = {}, []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    start = time.time()
    if name in param_grids:
        search = GridSearchCV(pipe, param_grids[name], scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
        search.fit(X_train, y_train)
        best, cv_mae, params = search.best_estimator_, -search.best_score_, search.best_params_
    else:
        cv_mae = -cross_val_score(pipe, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1).mean()
        best, params = pipe.fit(X_train, y_train), 'Configuración definida'
    best_models[name] = best
    cv_rows.append([name, cv_mae, params, time.time()-start])

cv_results = pd.DataFrame(cv_rows, columns=['Modelo','MAE CV (s)','Mejores parámetros','Tiempo (s)']).sort_values('MAE CV (s)')
display(cv_results)


,Modelo,MAE CV (s),Mejores parámetros,Tiempo (s)
0,Regresión lineal,156.640869,Configuración definida,1.190413
3,SVR,157.095246,"{'model__C': 50, 'model__epsilon': 0.05, 'mode...",1.056792
4,MLP,160.463772,"{'model__alpha': 0.001, 'model__hidden_layer_s...",7.097747
5,Voting,163.441908,Configuración definida,0.318966
7,Boosting,167.825092,"{'model__l2_regularization': 0.1, 'model__lear...",3.460591
6,Bagging,168.218928,"{'model__max_features': 1.0, 'model__max_sampl...",2.909536
2,Árbol de decisión,175.649003,"{'model__max_depth': 5, 'model__min_samples_le...",0.473576
1,KNN,232.740220,"{'model__n_neighbors': 7, 'model__p': 2, 'mode...",1.343982


## 4.3 Medida de calidad del modelo

### 4.3.1 Evaluación con el set de pruebas

Los modelos ya seleccionados se evalúan una sola vez sobre el 30 % reservado. Esto proporciona una estimación independiente de generalización.


In [10]:
test_rows = []
predictions = {}
for name, model in best_models.items():
    pred = model.predict(X_test); predictions[name] = pred
    test_rows.append([name, mean_absolute_error(y_test,pred),
                      np.sqrt(mean_squared_error(y_test,pred)), r2_score(y_test,pred)])

test_results = pd.DataFrame(test_rows, columns=['Modelo','MAE (s)','RMSE (s)','R²']).sort_values('MAE (s)')
display(test_results)
plt.figure(figsize=(9,4)); plt.barh(test_results['Modelo'], test_results['MAE (s)'])
plt.gca().invert_yaxis(); plt.xlabel('MAE (segundos)'); plt.title('Comparación en prueba'); plt.tight_layout(); plt.show()


,Modelo,MAE (s),RMSE (s),R²
3,SVR,148.438293,192.538022,0.867110
0,Regresión lineal,149.376529,192.975158,0.866506
4,MLP,151.672215,197.211807,0.860580
6,Bagging,155.612378,200.337551,0.856126
5,Voting,156.283879,201.193876,0.854893
7,Boosting,159.618663,206.451839,0.847210
2,Árbol de decisión,171.582892,218.440047,0.828950
1,KNN,209.460752,271.015622,0.736703


### 4.3.2 Selección del modelo con mejor desempeño

El modelo final se selecciona exclusivamente por el menor MAE en prueba, de acuerdo con la métrica principal definida antes de observar los resultados. La celda siguiente genera automáticamente la conclusión para evitar inconsistencias si el cuaderno se vuelve a ejecutar.


In [11]:
winner = test_results.iloc[0]
print(f"Modelo seleccionado: {winner['Modelo']}")
print(f"MAE: {winner['MAE (s)']:.2f} segundos ({winner['MAE (s)']/60:.2f} minutos)")
print(f"RMSE: {winner['RMSE (s)']:.2f} segundos")
print(f"R²: {winner['R²']:.4f}")

plt.figure(figsize=(6,5))
plt.scatter(y_test, predictions[winner['Modelo']], alpha=.45)
limits=[min(y_test.min(), predictions[winner['Modelo']].min()), max(y_test.max(), predictions[winner['Modelo']].max())]
plt.plot(limits, limits, 'r--'); plt.xlabel('Duración real (s)'); plt.ylabel('Predicción (s)')
plt.title(f"Valores reales vs. predichos — {winner['Modelo']}"); plt.tight_layout(); plt.show()


Modelo seleccionado: SVR
MAE: 148.44 segundos (2.47 minutos)
RMSE: 192.54 segundos
R²: 0.8671


## Conclusión

El flujo completo evita fuga de información, corrige los errores detectados, utiliza variables disponibles al inicio del viaje y compara ocho métodos bajo el mismo esquema de validación. El MAE permite traducir el desempeño final a segundos y minutos. Como limitación, el dataset no incluye tráfico en tiempo real, cierres viales ni incidentes, factores que podrían explicar parte del error residual.

**Enlace de Google Colab:** pegar aquí el enlace compartido con permiso “Cualquier persona que tenga el enlace”.


# SECCIÓN 5. DESPLIEGUE DEL MODELO CON STREAMLIT

Como etapa final, el modelo seleccionado se integró en una aplicación web construida con Streamlit y preparada para Streamlit Community Cloud. La aplicación convierte el análisis académico en una herramienta interactiva que puede ser utilizada sin ejecutar código.

**Repositorio de la aplicación:** https://github.com/StevenCu07/Taxi_Drip

La aplicación incorpora cuatro módulos:

1. **Predicción individual:** permite ingresar distancia en kilómetros o millas, pasajeros, clima, fecha, hora y coordenadas para estimar la duración.
2. **Procesamiento masivo:** acepta nuevos archivos CSV, realiza conversión bidireccional entre kilómetros y millas, genera predicciones y permite descargar los resultados.
3. **Comparación con datos reales:** cuando el archivo contiene `trip_duration`, calcula MAE, RMSE y R² y genera un gráfico real frente a predicho.
4. **Exploración y transparencia:** presenta las distribuciones, las variables, las limitaciones y permite seleccionar cualquiera de los ocho modelos evaluados. SVR aparece como recomendado por ser el ganador según MAE.

El despliegue utiliza los mismos pipelines y configuraciones de la sección 4. Los ocho modelos se entrenan al iniciar la aplicación y Streamlit los conserva en caché. SVR (`C=50`, `epsilon=0.05`, kernel lineal) continúa siendo el modelo oficial y predeterminado, mientras que los demás se ofrecen para comparación interactiva.


In [12]:
# Ejemplo de la lógica utilizada por la aplicación para una predicción
ejemplo_despliegue = X_test.iloc[[0]].copy()
prediccion_despliegue = best_models['SVR'].predict(ejemplo_despliegue)[0]

print('Variables recibidas por la aplicación:')
display(ejemplo_despliegue)
print(f'Duración estimada: {prediccion_despliegue:.2f} segundos')
print(f'Equivalente: {prediccion_despliegue/60:.2f} minutos')


Variables recibidas por la aplicación:


,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,trip_distance_miles,weather,pickup_hour,pickup_day_of_week,pickup_is_weekend
1388,40.752611,-74.042665,40.716391,-74.083951,NaN,5.559,Clear,9,5,1


Duración estimada: 1356.08 segundos
Equivalente: 22.60 minutos


## Archivos para el despliegue

- `app.py`: interfaz, entrenamiento, predicción individual, carga masiva y visualizaciones.
- `Taxi_Trip_Duration.csv`: datos utilizados para entrenar el pipeline.
- `requirements.txt`: versiones compatibles de Streamlit, pandas, NumPy, scikit-learn y Plotly.
- `.streamlit/config.toml`: configuración visual y del servidor.
- `Trabajo_final.ipynb`: cuaderno completo del proyecto.

## Procedimiento en Streamlit Community Cloud

1. Ingresar a https://share.streamlit.io/ con la cuenta de GitHub.
2. Seleccionar **Create app**.
3. Elegir el repositorio `StevenCu07/Taxi_Drip` y la rama `main`.
4. Seleccionar `app.py` como archivo principal.
5. Presionar **Deploy** y esperar la instalación de dependencias.

**Enlace público de la aplicación:** pegar aquí la URL generada por Streamlit Community Cloud.

La aplicación tiene un propósito académico. No incluye información de tráfico en tiempo real, incidentes ni cierres viales, por lo que sus resultados deben interpretarse como estimaciones y no como tiempos garantizados.


## 5.1 Generación automática de la aplicación

La siguiente celda crea físicamente todos los archivos de la aplicación en el entorno de Colab. El contenido de `app.py` es el mismo utilizado en el repositorio de GitHub: incluye los ocho modelos, SVR como opción recomendada, predicción individual, procesamiento masivo, métricas, visualizaciones y conversión bidireccional entre kilómetros y millas.

Antes de ejecutar, el archivo `Taxi_Trip_Duration.csv` debe aparecer en `/content`. Al finalizar, los archivos estarán visibles en el panel **Archivos** ubicado en el lado izquierdo de Colab.


In [13]:
from pathlib import Path
import json, shutil, py_compile, sys

# En Google Colab los archivos se generan directamente en /content.
# Fuera de Colab se usa una subcarpeta para no sobrescribir otros archivos.
IN_COLAB = 'google.colab' in sys.modules
APP_DIR = Path('/content') if IN_COLAB else Path('streamlit_app_generated')
APP_DIR.mkdir(parents=True, exist_ok=True)
(APP_DIR / ".streamlit").mkdir(parents=True, exist_ok=True)

APP_FILES = {'app.py': 'from __future__ import annotations\n\nfrom datetime import date, datetime, time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import BaggingRegressor, HistGradientBoostingRegressor, VotingRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.neighbors import KNeighborsRegressor\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.svm import SVR\nfrom sklearn.tree import DecisionTreeRegressor\n\n\nAPP_DIR = Path(__file__).parent\nDATA_PATH = APP_DIR / "Taxi_Trip_Duration.csv"\nMODEL_FEATURES = [\n    "pickup_latitude",\n    "pickup_longitude",\n    "dropoff_latitude",\n    "dropoff_longitude",\n    "passenger_count",\n    "trip_distance_miles",\n    "pickup_hour",\n    "pickup_day_of_week",\n    "pickup_is_weekend",\n    "weather",\n]\nNUMERIC_FEATURES = MODEL_FEATURES[:-1]\nWEATHER_OPTIONS = ["Clear", "Rain", "Snow", "Fog"]\n\nMODEL_RESULTS = pd.DataFrame(\n    [\n        ["SVR", 148.44, 192.54, 0.8671],\n        ["Regresión lineal", 149.38, 192.98, 0.8665],\n        ["MLP", 151.67, 197.21, 0.8606],\n        ["Bagging", 155.61, 200.34, 0.8561],\n        ["Voting", 156.28, 201.19, 0.8549],\n        ["Boosting", 159.62, 206.45, 0.8472],\n        ["Árbol de decisión", 171.58, 218.44, 0.8290],\n        ["KNN", 209.46, 271.02, 0.7367],\n    ],\n    columns=["Modelo", "MAE", "RMSE", "R²"],\n)\n\n\nst.set_page_config(\n    page_title="TaxiDuration AI",\n    page_icon="🚕",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\nst.markdown(\n    """\n    <style>\n    :root { --ink:#132238; --muted:#607086; --blue:#2165d6; --cyan:#27b4d1; --gold:#ffb000; }\n    .stApp { background: linear-gradient(180deg, #f6f9ff 0%, #ffffff 42%); }\n    .block-container { max-width: 1180px; padding-top: 1.8rem; padding-bottom: 3rem; }\n    [data-testid="stSidebar"] { background: #101b2f; }\n    [data-testid="stSidebar"] * { color: #f8fbff; }\n    .hero {\n      padding: 2.1rem 2.2rem; border-radius: 24px; color: white; margin-bottom: 1.2rem;\n      background: radial-gradient(circle at 85% 20%, rgba(39,180,209,.45), transparent 32%),\n                  linear-gradient(125deg, #101b2f 5%, #17498d 58%, #2165d6 100%);\n      box-shadow: 0 18px 45px rgba(24,73,141,.22);\n    }\n    .hero h1 { margin:0; font-size:2.45rem; letter-spacing:-.04em; }\n    .hero p { max-width:720px; margin:.7rem 0 0; color:#deebff; font-size:1.04rem; }\n    .eyebrow { color:#7de3f3; font-weight:700; letter-spacing:.14em; font-size:.76rem; text-transform:uppercase; }\n    .result-card {\n      background:linear-gradient(135deg,#132238,#2165d6); color:white; padding:1.4rem 1.6rem;\n      border-radius:20px; box-shadow:0 12px 28px rgba(22,65,130,.2); text-align:center;\n    }\n    .result-card .value { font-size:2.4rem; font-weight:800; margin:.15rem 0; }\n    .result-card .small { color:#d6e6ff; font-size:.9rem; }\n    .soft-card { background:white; border:1px solid #e4ebf5; border-radius:18px; padding:1rem 1.15rem; }\n    div[data-testid="stMetric"] { background:white; border:1px solid #e1e8f2; padding:14px 16px; border-radius:16px; }\n    [data-testid="stSidebar"] div[data-testid="stMetric"] {\n      background:rgba(255,255,255,.075); border:1px solid rgba(255,255,255,.16);\n      box-shadow:none;\n    }\n    [data-testid="stSidebar"] [data-testid="stMetricLabel"] p,\n    [data-testid="stSidebar"] [data-testid="stMetricValue"] div {\n      color:#f8fbff !important;\n    }\n    [data-testid="stSidebar"] [data-baseweb="select"] > div {\n      background:#1b2b45 !important;\n      border-color:rgba(255,255,255,.28) !important;\n    }\n    [data-testid="stSidebar"] [data-baseweb="select"] span,\n    [data-testid="stSidebar"] [data-baseweb="select"] input {\n      color:#ffffff !important;\n      -webkit-text-fill-color:#ffffff !important;\n    }\n    [data-testid="stSidebar"] [data-baseweb="select"] svg {\n      fill:#ffffff !important;\n    }\n    .active-model {\n      margin:.25rem 0 .65rem; padding:.65rem .75rem; border-radius:12px;\n      background:rgba(39,180,209,.13); border:1px solid rgba(125,227,243,.28);\n      color:#f8fbff; font-size:.88rem;\n    }\n    .active-model strong { color:#7de3f3; }\n    .stButton>button, .stDownloadButton>button { border-radius:12px; font-weight:700; }\n    h2, h3 { color:var(--ink); letter-spacing:-.02em; }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\n\n@st.cache_data\ndef load_and_prepare_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:\n    raw = pd.read_csv(DATA_PATH)\n    selected = raw.drop(\n        columns=[\n            "RecordID", "FullName", "Phone", "ZodiacSign", "FavoriteColor", "Hobby",\n            "total_fare", "VendorID", "trip_distance",\n        ],\n        errors="ignore",\n    ).copy()\n    valid = (\n        selected["trip_duration"].between(60, 14_400)\n        & selected["trip_distance_miles"].gt(0)\n        & selected["trip_distance_miles"].le(100)\n        & (\n            selected["passenger_count"].isna()\n            | (\n                selected["passenger_count"].between(1, 6)\n                & selected["passenger_count"].mod(1).eq(0)\n            )\n        )\n    )\n    clean = selected.loc[valid].copy()\n    dt = pd.to_datetime(clean["pickup_datetime"], dayfirst=True, errors="coerce")\n    clean["pickup_hour"] = dt.dt.hour\n    clean["pickup_day_of_week"] = dt.dt.dayofweek\n    clean["pickup_is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)\n    clean = clean.drop(columns=["pickup_datetime", "hour", "day_of_week", "is_weekend"])\n    return raw, clean[MODEL_FEATURES], clean["trip_duration"]\n\n\ndef build_preprocessor() -> ColumnTransformer:\n    numeric_pipe = Pipeline(\n        [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]\n    )\n    categorical_pipe = Pipeline(\n        [\n            ("imputer", SimpleImputer(strategy="constant", fill_value="Desconocido")),\n            ("onehot", OneHotEncoder(handle_unknown="ignore")),\n        ]\n    )\n    return ColumnTransformer(\n        [("num", numeric_pipe, NUMERIC_FEATURES), ("cat", categorical_pipe, ["weather"])]\n    )\n\n\ndef model_catalog() -> dict[str, object]:\n    return {\n        "SVR (recomendado)": SVR(C=50, epsilon=0.05, kernel="linear"),\n        "Regresión lineal": LinearRegression(),\n        "MLP": MLPRegressor(\n            hidden_layer_sizes=(100,), alpha=0.001, learning_rate_init=0.01,\n            max_iter=1000, early_stopping=True, random_state=42,\n        ),\n        "Bagging": BaggingRegressor(\n            estimator=DecisionTreeRegressor(random_state=42), n_estimators=100,\n            max_samples=0.5, max_features=1.0, random_state=42,\n        ),\n        "Voting": VotingRegressor(\n            [\n                ("lr", LinearRegression()),\n                ("tree", DecisionTreeRegressor(max_depth=5, random_state=42)),\n                ("hgb", HistGradientBoostingRegressor(random_state=42)),\n            ]\n        ),\n        "Boosting": HistGradientBoostingRegressor(\n            learning_rate=0.03, max_iter=100, max_leaf_nodes=15,\n            l2_regularization=0.1, random_state=42,\n        ),\n        "Árbol de decisión": DecisionTreeRegressor(\n            max_depth=5, min_samples_leaf=4, min_samples_split=2, random_state=42,\n        ),\n        "KNN": KNeighborsRegressor(n_neighbors=7, weights="distance", p=2),\n    }\n\n\n@st.cache_resource\ndef train_models() -> tuple[dict[str, Pipeline], dict[str, dict[str, float]]]:\n    _, x, y = load_and_prepare_data()\n    x_train, x_test, y_train, y_test = train_test_split(\n        x, y, test_size=0.30, random_state=42\n    )\n    trained: dict[str, Pipeline] = {}\n    metrics: dict[str, dict[str, float]] = {}\n    for name, estimator in model_catalog().items():\n        pipeline = Pipeline([("preprocessor", build_preprocessor()), ("model", estimator)])\n        pipeline.fit(x_train, y_train)\n        prediction = pipeline.predict(x_test)\n        trained[name] = pipeline\n        metrics[name] = {\n            "mae": float(mean_absolute_error(y_test, prediction)),\n            "rmse": float(np.sqrt(mean_squared_error(y_test, prediction))),\n            "r2": float(r2_score(y_test, prediction)),\n        }\n    return trained, metrics\n\n\ndef seconds_to_text(seconds: float) -> str:\n    rounded = max(0, int(round(seconds)))\n    hours, remainder = divmod(rounded, 3600)\n    minutes, secs = divmod(remainder, 60)\n    if hours:\n        return f"{hours} h {minutes} min"\n    return f"{minutes} min {secs} s"\n\n\ndef prepare_uploaded_data(\n    upload: pd.DataFrame, reference: pd.DataFrame\n) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:\n    data = upload.copy()\n    notes: list[str] = []\n    data.columns = data.columns.str.strip()\n\n    if "trip_distance_miles" not in data and "trip_distance" in data:\n        data["trip_distance_miles"] = pd.to_numeric(data["trip_distance"], errors="coerce") / 1.609344\n        notes.append("Se convirtió `trip_distance` de kilómetros a millas.")\n    if "trip_distance_miles" not in data:\n        raise ValueError("El archivo debe incluir `trip_distance_miles` o `trip_distance`.")\n    if "trip_distance" not in data:\n        data["trip_distance"] = pd.to_numeric(data["trip_distance_miles"], errors="coerce") * 1.609344\n        notes.append("Se convirtió `trip_distance_miles` de millas a kilómetros.")\n\n    if "pickup_datetime" in data:\n        dt = pd.to_datetime(data["pickup_datetime"], dayfirst=True, errors="coerce")\n        data["pickup_hour"] = dt.dt.hour\n        data["pickup_day_of_week"] = dt.dt.dayofweek\n        data["pickup_is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(float)\n        notes.append("Las variables temporales se generaron desde `pickup_datetime`.")\n\n    defaults = {\n        **{column: float(reference[column].median()) for column in NUMERIC_FEATURES},\n        "weather": str(reference["weather"].mode().iloc[0]),\n    }\n    missing = [column for column in MODEL_FEATURES if column not in data]\n    for column in missing:\n        data[column] = defaults[column]\n    if missing:\n        notes.append("Campos ausentes completados con valores típicos: " + ", ".join(missing) + ".")\n\n    for column in NUMERIC_FEATURES:\n        data[column] = pd.to_numeric(data[column], errors="coerce")\n    data["weather"] = data["weather"].astype("string").fillna("Desconocido")\n    return data[MODEL_FEATURES], notes, data\n\n\nraw_data, model_x, model_y = load_and_prepare_data()\ntrained_models, all_model_metrics = train_models()\n\nwith st.sidebar:\n    st.markdown("## 🚕 TaxiDuration AI")\n    st.caption("Predicción y análisis de duración de viajes")\n    st.markdown("---")\n    st.markdown("**Modelo activo**")\n    selected_model = st.selectbox(\n        "Selecciona un modelo",\n        list(trained_models),\n        help="SVR es el modelo oficial porque obtuvo el menor MAE en el trabajo final.",\n        label_visibility="collapsed",\n    )\n    st.markdown(\n        f\'<div class="active-model">Modelo seleccionado:<br><strong>{selected_model}</strong></div>\',\n        unsafe_allow_html=True,\n    )\n    model = trained_models[selected_model]\n    model_metrics = all_model_metrics[selected_model]\n    if selected_model == "SVR (recomendado)":\n        st.success("✓ Ganador del trabajo final")\n    else:\n        st.info("Modelo alternativo para comparación")\n    st.metric("MAE de referencia", f"{model_metrics[\'mae\']:.1f} s")\n    st.metric("R²", f"{model_metrics[\'r2\']:.3f}")\n    st.markdown("---")\n    st.caption("Proyecto final · Metodología CRISP-DM")\n    st.caption("Brandon Steven Calzada Urrea · Juan Felipe Giraldo")\n\nst.markdown(\n    """\n    <section class="hero">\n      <div class="eyebrow">Machine learning aplicado a movilidad</div>\n      <h1>Predice cuánto durará tu viaje</h1>\n      <p>Una experiencia interactiva para estimar trayectos, analizar nuevos archivos y comparar el desempeño del modelo con datos reales.</p>\n    </section>\n    """,\n    unsafe_allow_html=True,\n)\n\ntab_predict, tab_batch, tab_explore, tab_model = st.tabs(\n    ["✨ Predicción individual", "📂 Analizar archivo", "📊 Explorar datos", "🧠 Acerca del modelo"]\n)\n\nwith tab_predict:\n    st.subheader("Configura un viaje")\n    st.caption("Completa las condiciones conocidas antes de iniciar el recorrido.")\n    with st.form("single_prediction"):\n        left, middle, right = st.columns(3)\n        with left:\n            distance_unit = st.radio("Unidad de distancia", ["Kilómetros", "Millas"], horizontal=True)\n            if distance_unit == "Kilómetros":\n                distance = st.number_input("Distancia estimada (km)", min_value=0.1, max_value=160.9, value=4.8, step=0.1)\n            else:\n                distance = st.number_input("Distancia estimada (millas)", min_value=0.1, max_value=100.0, value=3.0, step=0.1)\n            passengers = st.number_input("Número de pasajeros", min_value=1, max_value=6, value=1)\n            weather = st.selectbox("Clima", WEATHER_OPTIONS)\n        with middle:\n            travel_date = st.date_input("Fecha del viaje", value=date(2023, 6, 15))\n            travel_time = st.time_input("Hora de recogida", value=time(12, 0))\n            st.caption("Las coordenadas permiten incorporar el contexto espacial del trayecto.")\n        with right:\n            pickup_lat = st.number_input("Latitud de origen", value=40.7500, format="%.6f")\n            pickup_lon = st.number_input("Longitud de origen", value=-73.9800, format="%.6f")\n            dropoff_lat = st.number_input("Latitud de destino", value=40.7700, format="%.6f")\n            dropoff_lon = st.number_input("Longitud de destino", value=-73.9500, format="%.6f")\n        submitted = st.form_submit_button("Estimar duración", type="primary", width="stretch")\n\n    if submitted:\n        trip_dt = datetime.combine(travel_date, travel_time)\n        distance_miles = distance / 1.609344 if distance_unit == "Kilómetros" else distance\n        distance_km = distance * 1.609344 if distance_unit == "Millas" else distance\n        row = pd.DataFrame(\n            [{\n                "pickup_latitude": pickup_lat, "pickup_longitude": pickup_lon,\n                "dropoff_latitude": dropoff_lat, "dropoff_longitude": dropoff_lon,\n                "passenger_count": passengers, "trip_distance_miles": distance_miles,\n                "pickup_hour": trip_dt.hour, "pickup_day_of_week": trip_dt.weekday(),\n                "pickup_is_weekend": int(trip_dt.weekday() >= 5), "weather": weather,\n            }]\n        )\n        estimate = float(model.predict(row)[0])\n        low = max(60.0, estimate - model_metrics["mae"])\n        high = estimate + model_metrics["mae"]\n        c1, c2 = st.columns([1.1, 1])\n        with c1:\n            st.markdown(\n                f"""<div class="result-card"><div class="small">Duración estimada</div>\n                <div class="value">{seconds_to_text(estimate)}</div><div>{estimate:,.0f} segundos</div>\n                <div class="small">Distancia: {distance_km:.2f} km · {distance_miles:.2f} mi</div>\n                <div class="small" style="margin-top:.65rem">Rango orientativo: {seconds_to_text(low)} – {seconds_to_text(high)}</div></div>""",\n                unsafe_allow_html=True,\n            )\n        with c2:\n            st.info(\n                f"Predicción generada con **{selected_model}**. El rango usa su MAE de prueba como referencia; no es un intervalo de confianza."\n            )\n\n    with st.expander("↔️ Conversor rápido de distancia"):\n        convert_from = st.radio("Convertir desde", ["Kilómetros", "Millas"], horizontal=True, key="converter_unit")\n        convert_value = st.number_input("Valor", min_value=0.0, value=10.0, step=0.1, key="converter_value")\n        if convert_from == "Kilómetros":\n            st.markdown(f"**{convert_value:.2f} km = {convert_value / 1.609344:.2f} millas**")\n        else:\n            st.markdown(f"**{convert_value:.2f} millas = {convert_value * 1.609344:.2f} km**")\n\nwith tab_batch:\n    st.subheader("Predicción masiva y comparación")\n    st.write(\n        "Carga un CSV con `trip_distance_miles` o `trip_distance`. Si también contiene `trip_duration`, la app calculará métricas contra los valores reales."\n    )\n    with st.expander("Ver columnas recomendadas"):\n        st.code(\n            "pickup_datetime, pickup_latitude, pickup_longitude, dropoff_latitude, "\n            "dropoff_longitude, passenger_count, trip_distance_miles, weather, trip_duration",\n            language="text",\n        )\n        sample = raw_data[[\n            "pickup_datetime", "pickup_latitude", "pickup_longitude", "dropoff_latitude",\n            "dropoff_longitude", "passenger_count", "trip_distance_miles", "weather", "trip_duration",\n        ]].head(20)\n        st.download_button(\n            "Descargar CSV de ejemplo", sample.to_csv(index=False).encode("utf-8"),\n            "ejemplo_viajes_taxi.csv", "text/csv",\n        )\n\n    uploaded = st.file_uploader("Arrastra un archivo CSV", type=["csv"])\n    if uploaded is not None:\n        try:\n            uploaded_df = pd.read_csv(uploaded)\n            prepared, preparation_notes, output = prepare_uploaded_data(uploaded_df, model_x)\n            output["predicted_trip_duration"] = model.predict(prepared)\n            output["predicted_duration_minutes"] = output["predicted_trip_duration"] / 60\n            st.success(f"Archivo procesado correctamente: {len(output):,} registros.")\n            st.caption(f"Predicciones generadas con: **{selected_model}**")\n            for note in preparation_notes:\n                st.caption("• " + note)\n\n            if "trip_duration" in output:\n                valid_target = pd.to_numeric(output["trip_duration"], errors="coerce").notna()\n                actual = pd.to_numeric(output.loc[valid_target, "trip_duration"])\n                predicted = output.loc[valid_target, "predicted_trip_duration"]\n                if len(actual) >= 2:\n                    mae = mean_absolute_error(actual, predicted)\n                    rmse = np.sqrt(mean_squared_error(actual, predicted))\n                    r2 = r2_score(actual, predicted)\n                    m1, m2, m3 = st.columns(3)\n                    m1.metric("MAE del archivo", f"{mae:,.1f} s", delta=f"{model_metrics[\'mae\']-mae:+.1f} vs. referencia")\n                    m2.metric("RMSE", f"{rmse:,.1f} s")\n                    m3.metric("R²", f"{r2:.3f}")\n                    comparison = pd.DataFrame({"Real": actual, "Predicción": predicted})\n                    fig = px.scatter(\n                        comparison, x="Real", y="Predicción", opacity=0.55,\n                        title="Duración real frente a predicción", template="plotly_white",\n                    )\n                    max_value = float(comparison.max().max())\n                    fig.add_shape(type="line", x0=0, y0=0, x1=max_value, y1=max_value, line=dict(dash="dash", color="#e74c3c"))\n                    st.plotly_chart(fig, width="stretch")\n            st.dataframe(output.head(200), width="stretch", hide_index=True)\n            st.download_button(\n                "Descargar predicciones", output.to_csv(index=False).encode("utf-8"),\n                "predicciones_taxi.csv", "text/csv", type="primary",\n            )\n        except Exception as exc:\n            st.error(f"No fue posible procesar el archivo: {exc}")\n\nwith tab_explore:\n    st.subheader("Radiografía del conjunto de datos")\n    e1, e2, e3, e4 = st.columns(4)\n    e1.metric("Registros originales", f"{len(raw_data):,}")\n    e2.metric("Registros modelados", f"{len(model_x):,}")\n    e3.metric("Distancia mediana", f"{model_x.trip_distance_miles.median():.2f} mi")\n    e4.metric("Duración mediana", seconds_to_text(model_y.median()))\n\n    c1, c2 = st.columns(2)\n    with c1:\n        fig = px.histogram(\n            pd.DataFrame({"Duración (min)": model_y / 60}), x="Duración (min)", nbins=45,\n            title="Distribución de la duración", color_discrete_sequence=["#2165d6"], template="plotly_white",\n        )\n        st.plotly_chart(fig, width="stretch")\n    with c2:\n        relation = pd.DataFrame({"Distancia (mi)": model_x.trip_distance_miles, "Duración (min)": model_y / 60})\n        fig = px.scatter(\n            relation, x="Distancia (mi)", y="Duración (min)", opacity=0.4,\n            title="Distancia frente a duración", color_discrete_sequence=["#27b4d1"], template="plotly_white",\n        )\n        st.plotly_chart(fig, width="stretch")\n\n    weather_df = pd.DataFrame({"Clima": model_x.weather.fillna("Desconocido"), "Duración (min)": model_y / 60})\n    fig = px.box(\n        weather_df, x="Clima", y="Duración (min)", color="Clima",\n        title="Duración según condición climática", template="plotly_white",\n    )\n    st.plotly_chart(fig, width="stretch")\n\nwith tab_model:\n    st.subheader("Modelo, evaluación y transparencia")\n    st.write(\n        "El flujo aplica CRISP-DM: selección por conocimiento del negocio, limpieza de errores, ingeniería temporal, preprocesamiento sin fuga de información, validación cruzada y evaluación independiente."\n    )\n    m1, m2, m3, m4 = st.columns(4)\n    m1.metric("Modelo seleccionado", selected_model.replace(" (recomendado)", ""))\n    m2.metric("MAE", f"{model_metrics[\'mae\']:.2f} s")\n    m3.metric("RMSE", f"{model_metrics[\'rmse\']:.2f} s")\n    m4.metric("R²", f"{model_metrics[\'r2\']:.4f}")\n\n    fig = px.bar(\n        MODEL_RESULTS.sort_values("MAE", ascending=False), x="MAE", y="Modelo", orientation="h",\n        color="MAE", color_continuous_scale=["#27b4d1", "#2165d6", "#132238"],\n        title="MAE de los ocho modelos en el conjunto de prueba", template="plotly_white",\n    )\n    fig.update_coloraxes(showscale=False)\n    st.plotly_chart(fig, width="stretch")\n\n    st.info(\n        "SVR permanece como modelo oficial del trabajo porque logró el menor MAE. "\n        "El selector lateral permite usar los otros siete modelos con fines comparativos, sin cambiar la conclusión académica."\n    )\n\n    col_a, col_b = st.columns(2)\n    with col_a:\n        st.markdown("#### Variables utilizadas")\n        st.markdown(\n            "- Coordenadas de origen y destino\\n- Pasajeros y distancia estimada\\n"\n            "- Hora, día de la semana y fin de semana\\n- Condición climática"\n        )\n    with col_b:\n        st.markdown("#### Salvaguardas metodológicas")\n        st.markdown(\n            "- `total_fare` excluida por fuga de información\\n- Imputación aprendida solo con entrenamiento\\n"\n            "- One-Hot Encoding para el clima\\n- Prueba independiente del 30 %"\n        )\n    st.warning(\n        "Limitación: el modelo no recibe tráfico en tiempo real, obras, cierres ni incidentes. La estimación es académica y no sustituye un sistema de navegación."\n    )\n', 'requirements.txt': 'streamlit>=1.42,<2\npandas>=2.2,<3\nnumpy>=1.26,<3\nscikit-learn>=1.5,<2\nplotly>=5.24,<7\n\n', '.streamlit/config.toml': '[theme]\nprimaryColor = "#2165D6"\nbackgroundColor = "#F7F9FD"\nsecondaryBackgroundColor = "#FFFFFF"\ntextColor = "#132238"\nfont = "sans serif"\n\n[server]\nheadless = true\nmaxUploadSize = 200\n\n[browser]\ngatherUsageStats = false\n\n', 'README.md': '# TaxiDuration AI 🚕\n\nAplicación interactiva de Machine Learning para predecir la duración de viajes en taxi. El proyecto utiliza un modelo **SVR** seleccionado después de comparar cinco modelos clásicos y tres ensambles bajo la metodología CRISP-DM.\n\n## Funcionalidades\n\n- Predicción individual con distancia, pasajeros, clima, fecha, hora y coordenadas.\n- Selector interactivo entre los ocho modelos evaluados; SVR aparece como recomendado.\n- Conversión bidireccional entre kilómetros y millas.\n- Procesamiento masivo de archivos CSV.\n- Conversión automática de kilómetros a millas.\n- Imputación segura de campos opcionales.\n- Comparación con valores reales cuando el archivo incluye `trip_duration`.\n- Descarga de las predicciones generadas.\n- Panel exploratorio del dataset.\n- Comparación de los ocho modelos evaluados.\n\n## Ejecución local\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n## Despliegue en Streamlit Community Cloud\n\n1. Ingresar a [share.streamlit.io](https://share.streamlit.io/).\n2. Seleccionar **Create app**.\n3. Elegir el repositorio `StevenCu07/Taxi_Drip`, rama `main`.\n4. Indicar `app.py` como archivo principal.\n5. Presionar **Deploy**.\n\n## Archivos principales\n\n- `app.py`: aplicación Streamlit.\n- `Taxi_Trip_Duration.csv`: dataset usado para entrenar el modelo.\n- `Taxi_Trip_Duration_Info.txt`: descripción de variables y problemas de calidad intencionales del dataset.\n- `Trabajo_final.ipynb`: cuaderno completo de preparación, modelamiento y evaluación.\n- `requirements.txt`: dependencias para el despliegue.\n\n## Resultados del modelo seleccionado\n\n| Métrica | Resultado |\n|---|---:|\n| MAE | 148,44 segundos |\n| RMSE | 192,54 segundos |\n| R² | 0,8671 |\n\nProyecto académico desarrollado por Brandon Steven Calzada Urrea y Juan Felipe Giraldo.\n', 'Taxi_Trip_Duration_Info.txt': 'Taxi trip records in a city. Goal: predict trip duration.\n\nINTENTIONAL DATA QUALITY ISSUES:\n- pickup_datetime as text (DD/MM/YYYY) — convert to datetime and engineer features\n- Missing values in trip_distance, passenger_count, weather, pickup_latitude\n- Outliers in distance, duration, passenger_count\n- passenger_count as float\n- Noise/ID columns: RecordID, FullName, Phone, ZodiacSign, FavoriteColor, Hobby\n- Redundant: trip_distance_miles vs trip_distance\n- Near-constant: VendorID\n- TARGET LEAKAGE: total_fare known after/with trip pricing — remove for pure duration prediction\n- Different scales (coords vs distance vs duration)\n\n* DATA (predictors):\n- RecordID, FullName, Phone, ZodiacSign, FavoriteColor, Hobby: irrelevant\n- pickup_datetime: Pickup timestamp as text\n- pickup_latitude, pickup_longitude: Pickup coordinates\n- dropoff_latitude, dropoff_longitude: Dropoff coordinates\n- passenger_count: Number of passengers (float for type practice)\n- trip_distance: Distance in km\n- trip_distance_miles: Distance in miles (redundant)\n- hour, day_of_week, is_weekend: Time features\n- weather: Clear / Rain / Snow / Fog\n- VendorID: Taxi vendor (near-constant)\n- total_fare: Fare amount (TARGET LEAKAGE risk — remove)\n\n* TARGET:\n- trip_duration: Trip duration in seconds (regression)\n'}

for relative_path, contents in APP_FILES.items():
    destination = APP_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding='utf-8')

dataset_source = Path('/content/Taxi_Trip_Duration.csv') if IN_COLAB else Path('Taxi_Trip_Duration.csv')
if not dataset_source.exists() and Path('../Taxi_Trip_Duration.csv').exists():
    dataset_source = Path('../Taxi_Trip_Duration.csv')
if not dataset_source.exists():
    raise FileNotFoundError('Debes subir Taxi_Trip_Duration.csv antes de ejecutar esta celda.')
dataset_destination = APP_DIR / "Taxi_Trip_Duration.csv"
if dataset_source.resolve() != dataset_destination.resolve():
    shutil.copy2(dataset_source, dataset_destination)

py_compile.compile(str(APP_DIR / 'app.py'), cfile=str(Path('/tmp') / 'taxi_app_validation.pyc'), doraise=True)
required = ['app.py','requirements.txt','.streamlit/config.toml','README.md',
            'Taxi_Trip_Duration_Info.txt','Taxi_Trip_Duration.csv']
missing = [name for name in required if not (APP_DIR/name).exists()]
assert not missing, f'Faltan archivos: {missing}'

print(f'Aplicación generada correctamente en: {APP_DIR.resolve()}')
print('Archivos creados:')
for name in required:
    print(f'  ✓ {name}')
print('✓ app.py compila sin errores')


Aplicación generada correctamente en: /home/brandon/Documents/Universidad/Seminario/Modulo 3/Final/streamlit_app_generated
Archivos creados:
  ✓ app.py
  ✓ requirements.txt
  ✓ .streamlit/config.toml
  ✓ README.md
  ✓ Taxi_Trip_Duration_Info.txt
  ✓ Taxi_Trip_Duration.csv
✓ app.py compila sin errores


## 5.2 Empaquetado para Drive o GitHub

Esta celda crea `Taxi_Drip_Streamlit.zip` con todos los archivos necesarios. El ZIP puede descargarse, guardarse en Google Drive o utilizarse como respaldo. Streamlit Community Cloud seguirá desplegando la aplicación desde GitHub.


In [14]:
import shutil, tempfile
from pathlib import Path

zip_path = APP_DIR / 'Taxi_Drip_Streamlit.zip'
if zip_path.exists():
    zip_path.unlink()
temporary_base = Path(tempfile.gettempdir()) / 'Taxi_Drip_Streamlit'
temporary_zip = Path(shutil.make_archive(str(temporary_base), 'zip', root_dir=APP_DIR,
                                         base_dir='.'))
shutil.copy2(temporary_zip, zip_path)
print(f'Paquete generado: {zip_path}')
print(f'Tamaño: {zip_path.stat().st_size / 1024:.1f} KB')

if IN_COLAB:
    print('El ZIP aparece en el panel Archivos de Colab y puede descargarse o copiarse a Drive.')


Paquete generado: streamlit_app_generated/Taxi_Drip_Streamlit.zip
Tamaño: 178.5 KB


## 5.3 Ejecución y despliegue

Para una prueba temporal dentro de Colab se puede abrir una terminal y ejecutar `streamlit run app.py`; sin embargo, ese proceso permanece activo y no se incluye en **Ejecutar todo** para evitar bloquear el cuaderno. El despliegue permanente se realiza desde https://share.streamlit.io/ seleccionando el repositorio `StevenCu07/Taxi_Drip`, la rama `main` y el archivo `app.py`.

**Aplicación pública:** pegar aquí la URL definitiva generada por Streamlit Community Cloud.
